# Deliverable 2 – Campus Coffee Shop
INF201 · Sigurd Aleksander Ruud and Shem Abdul-Rahman

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
#imports and gets all the packages needed for the code to run
DATA_DIR = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

## Task 1.1 Robust file reader
Try UTF-8 first. If it fails, fall back to ISO-8859-1.

In [2]:
def read_file(path) -> tuple[str, str]:
    """Read a text file as UTF-8, fall back to ISO-8859-1. Returns (text, encoding)."""
    path = Path(path)
    try:
        return path.read_text(encoding="utf-8"), "utf-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1"

## Task 1.2 Log parser
Regex captures the 6 fields. Noise lines don't match and are skipped.

In [3]:
LINE_PATTERN = re.compile(
    r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]"   # timestamp
    r" \[(\w+)\]"                                  # register_id
    r" item=(\w+)"                                 # item
    r" price=(\d+\.\d+)"                           # price
    r" ([A-Z]{3})"                                 # currency
    r" \((SUCCESS|REFUND|CANCELLED)\)"             # status
)


def parse_log(text: str) -> list[dict]:
    """Return one dict per valid sales line; noise lines are ignored."""
    records = []
    for line in text.splitlines():
        match = LINE_PATTERN.search(line)
        if match:
            ts, reg, item, price, cur, status = match.groups()
            records.append({
                "timestamp": ts,
                "register_id": reg,
                "item": item,
                "price": float(price),
                "currency": cur,
                "status": status,
            })
    return records

In [4]:
test = """[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
--- PRINTER PAPER LOW ---"""
parse_log(test)

[{'timestamp': '2026-10-15 08:30:00',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.5,
  'currency': 'EUR',
  'status': 'SUCCESS'}]

## Task 2.1 Recursive file search
The function calls itself on every sub-folder (no glob/rglob/os.walk).

In [5]:
def find_log_files(folder, extension: str = ".log") -> list[Path]:
    """Recursively find all files with the given extension."""
    folder = Path(folder)
    found = []
    for item in folder.iterdir():
        if item.is_dir():
            found.extend(find_log_files(item, extension))   # calls itself
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    return found

In [6]:
files = find_log_files(DATA_DIR)
len(files), files[0]

(9, PosixPath('data/2026/science_building/POS_01/register_sales.log'))

## Task 2.2 Recursive search for active registers
Handles dicts (check + search values), lists (search elements), and other values (stop).

In [7]:
def find_active_registers(data) -> set[str]:
    """Return the ids of all registers with "active": true, at any depth."""
    active = set()
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active.add(data["id"])
        for value in data.values():
            active |= find_active_registers(value)
    elif isinstance(data, list):
        for element in data:
            active |= find_active_registers(element)
    return active

In [8]:
config = json.loads((DATA_DIR / "store_config.json").read_text(encoding="utf-8"))
find_active_registers(config)

{'POS_01', 'POS_02', 'POS_03'}

## Task 3.1 Outlier removal
A value is an outlier if |x − μ| > 3σ. Outliers are replaced by the median of the valid prices.

In [ ]:
def remove_outliers(prices) -> np.ndarray:
    """Replace values more than 3 std from the mean with the median of the valid values."""
    x = np.asarray(prices, dtype=float).ravel()
    mu, sigma = x.mean(), x.std()
    is_outlier = np.abs(x - mu) > 3 * sigma
    cleaned = x.copy()
    cleaned[is_outlier] = np.median(x[~is_outlier])
    return cleaned

In [ ]:
remove_outliers([3, 4, 4.5, 3.8, 4.2, 3.5, 4.1, 3.9, 4.4, 3.6, 999])

## Task 3.2 Refund ratio
Refund ratio = refunds / (sales + 1e-6). The small epsilon avoids division by zero.